# 00 · Exploración inicial del dataset
**Responsable:** Santiago · **Módulo 0** · Proyecto: Perfiles musicales con aprendizaje no supervisado

**Objetivo.** Antes de limpiar o modelar nada, entender *qué tenemos*: cuántas canciones, qué columnas,
qué tipo de dato es cada una, cuántos nulos y duplicados hay y cuántos géneros trae el dataset.

**Fuente.** Spotify Tracks Dataset (Pandya, 2023), publicado en Hugging Face y Kaggle. Se usa el CSV público ya
publicado; **no** se consulta la API de Spotify directamente. Ver `data/raw/README.md`.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src import config as cfg
from src import preprocessing as pp
from src import viz

viz.set_style()
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:,.3f}".format)

## 1. Carga y dimensiones

In [2]:
raw = pp.load_raw()
print(f"{raw.shape[0]:,} filas x {raw.shape[1]} columnas")
raw.head()

114,000 filas x 20 columnas


,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.461,1,-6.746,0,0.143,0.032,0.000,0.358,0.715,87.917,4,acoustic
1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.166,1,-17.235,1,0.076,0.924,0.000,0.101,0.267,77.489,4,acoustic
2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.359,0,-9.734,1,0.056,0.210,0.000,0.117,0.120,76.332,4,acoustic
3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.060,0,-18.515,1,0.036,0.905,0.000,0.132,0.143,181.740,3,acoustic
4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.443,2,-9.681,1,0.053,0.469,0.000,0.083,0.167,119.949,4,acoustic


## 2. Tipos de dato y nulos

In [3]:
info = pd.DataFrame({"tipo": raw.dtypes.astype(str), "nulos": raw.isna().sum(),
                     "pct_nulos": 100 * raw.isna().mean(), "valores_unicos": raw.nunique()})
info

,tipo,nulos,pct_nulos,valores_unicos
track_id,str,0,0.000,89741
artists,str,1,0.001,31437
album_name,str,1,0.001,46589
track_name,str,1,0.001,73608
popularity,int64,0,0.000,101
duration_ms,int64,0,0.000,50697
explicit,bool,0,0.000,2
danceability,float64,0,0.000,1174
energy,float64,0,0.000,2083
key,int64,0,0.000,12


## 3. Duplicados y multigénero
Una misma canción (`track_id`) puede aparecer varias veces, una por cada género en el que fue listada.
Esto importa por dos razones: infla artificialmente el tamaño del dataset y **un mismo punto no puede
pertenecer a dos clusters a la vez**. Medimos cuánto pasa.

In [4]:
n_rows = len(raw)
n_tracks = raw[cfg.ID_COL].nunique()
print(f"Filas: {n_rows:,} | track_id únicos: {n_tracks:,} | filas repetidas por track_id: {n_rows - n_tracks:,}")

g = raw.groupby(cfg.ID_COL)[cfg.GENRE_COL].nunique()
print("\nGéneros por canción (distribución):")
print(g.value_counts().sort_index().head(10).to_string())

Filas: 114,000 | track_id únicos: 89,741 | filas repetidas por track_id: 24,259

Géneros por canción (distribución):
track_genre
1    73442
2    11424
3     2955
4     1361
5      431
6      104
7       21
8        2
9        1


## 4. Géneros

In [5]:
genre_counts = raw[cfg.GENRE_COL].value_counts()
print(f"{genre_counts.size} géneros distintos")
print(genre_counts.describe().to_string())
genre_counts.head(10)

114 géneros distintos


count     114.000
mean    1,000.000
std         0.000
min     1,000.000
25%     1,000.000
50%     1,000.000
75%     1,000.000
max     1,000.000


track_genre
acoustic       1000
afrobeat       1000
alt-rock       1000
alternative    1000
ambient        1000
anime          1000
black-metal    1000
bluegrass      1000
blues          1000
brazil         1000
Name: count, dtype: int64

## 5. Variables candidatas: rangos y valores sospechosos
Revisamos mínimos y máximos de las 10 variables acústicas elegidas y buscamos ceros que no tengan sentido
(un `tempo` de 0 o una duración de 0 no es música medible).

In [6]:
raw[cfg.FEATURES].describe().T

,count,mean,std,min,25%,50%,75%,max
danceability,"114,000.000",0.567,0.174,0.000,0.456,0.580,0.695,0.985
energy,"114,000.000",0.641,0.252,0.000,0.472,0.685,0.854,1.000
loudness,"114,000.000",-8.259,5.029,-49.531,-10.013,-7.004,-5.003,4.532
speechiness,"114,000.000",0.085,0.106,0.000,0.036,0.049,0.085,0.965
acousticness,"114,000.000",0.315,0.333,0.000,0.017,0.169,0.598,0.996
instrumentalness,"114,000.000",0.156,0.310,0.000,0.000,0.000,0.049,1.000
liveness,"114,000.000",0.214,0.190,0.000,0.098,0.132,0.273,1.000
valence,"114,000.000",0.474,0.259,0.000,0.260,0.464,0.683,0.995
tempo,"114,000.000",122.148,29.978,0.000,99.219,122.017,140.071,243.372
duration_ms,"114,000.000","228,029.153","107,297.713",0.000,"174,066.000","212,906.000","261,506.000","5,237,295.000"


In [7]:
sospechosos = pd.Series({
    "tempo == 0": int((raw["tempo"] == 0).sum()),
    "duration_ms == 0": int((raw["duration_ms"] == 0).sum()),
    "loudness > 0 dB": int((raw["loudness"] > 0).sum()),
})
sospechosos

tempo == 0          157
duration_ms == 0      1
loudness > 0 dB      90
dtype: int64

## 6. Hallazgos y decisiones
> **Completar en equipo tras ejecutar las celdas de arriba.** Responder con números:
> 1. ¿Cuántas canciones únicas hay realmente y qué % de filas son repeticiones por multigénero?
> 2. ¿Cuántos nulos y registros inválidos hay y en qué columnas?
> 3. ¿Los géneros están balanceados? ¿Cuál es el más y el menos frecuente?
> 4. Decisión: ¿trabajamos con todos los géneros o con un subconjunto? ¿Con qué criterio?